[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhiksark/pycon-italy-2026-workshop/blob/solutions/solutions/workshop-all.solution.ipynb)

# Write Your First High-Performance GPU Kernel in Python!

**PyCon Italy 2026 · Bologna** — all five workshop notebooks in one file.

1. *Runtime → Change runtime type → **T4 GPU***
2. *Runtime → Run all* (top to bottom), or run section by section.
3. *File → Save a copy in Drive* so your work survives a disconnect.

The five separate notebooks (`01`–`05`) remain the canonical versions; this is the
all-in-one convenience copy.

## Setup — run me first

In [ ]:
import importlib, subprocess, sys, os, pathlib

def ensure(pip_name, import_name=None):
    try:
        importlib.import_module(import_name or pip_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])

ensure('triton')
ensure('Pillow', 'PIL')
ensure('matplotlib')
ensure('opencv-python', 'cv2')

# cupy is used only by the Notebook 1 demo; keep it optional.
_cupy_ok = True
try:
    ensure('cupy-cuda12x', 'cupy')
    import cupy as cp
except Exception as exc:
    _cupy_ok = False
    print(f'cupy unavailable (only used in the Notebook 1 demo): {exc}')

# Clone the workshop repo (only for the blur image asset) and chdir in.
if not pathlib.Path('notebooks/shared').is_dir():
    repo_url = os.environ.get('WORKSHOP_REPO_URL',
                              'https://github.com/abhiksark/pycon-italy-2026-workshop.git')
    target = '/content/pycon-italy'
    if not pathlib.Path(target).exists():
        subprocess.check_call(['git', 'clone', '--depth=1', repo_url, target])
    os.chdir(target)

import torch, triton
import triton.language as tl
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import cv2

assert torch.cuda.is_available(), \
    'No CUDA GPU detected. On Colab: Runtime > Change runtime type > T4 GPU.'
print(f'triton {triton.__version__} | gpu {torch.cuda.get_device_name(0)}'
      + (f' | cupy {cp.__version__}' if _cupy_ok else ' | cupy: unavailable'))

## Timing helpers

In [ ]:
# Timing helpers — inlined from notebooks/shared/benchmark_utils.py so this notebook
# depends only on pip packages (no local repo imports). CUDA sync stays the caller's
# job: the workload callable must call torch.cuda.synchronize() before returning.
import time
from statistics import median
from typing import Callable


def gflops_for_matmul(m: int, n: int, k: int, seconds: float) -> float:
    """Returns sustained GFLOPs for an `(m, k) @ (k, n) -> (m, n)` matmul.

    A matmul performs `2 * m * n * k` floating-point operations (one multiply
    and one add per inner-product step).

    Args:
        m: rows of A and rows of the output.
        n: columns of B and columns of the output.
        k: shared inner dimension.
        seconds: wall-clock seconds the matmul took.
    """
    if seconds <= 0:
        raise ValueError(f"seconds must be positive, got {seconds!r}")
    return (2.0 * m * n * k) / (seconds * 1e9)


def median_seconds(
    workload: Callable[[], None],
    *,
    runs: int,
    warmup: int,
) -> float:
    """Times a no-argument callable and returns the median wall-clock seconds.

    The caller is responsible for any device synchronisation inside the
    workload. The first ``warmup`` calls are discarded; the next ``runs``
    are timed and their median is returned.

    Args:
        workload: A no-argument callable to time.
        runs: Number of timed iterations; must be positive.
        warmup: Number of untimed warm-up calls; must be non-negative.

    Returns:
        Median wall-clock duration in seconds across ``runs`` iterations.
    """
    if runs <= 0:
        raise ValueError(f"runs must be positive, got {runs!r}")
    if warmup < 0:
        raise ValueError(f"warmup must be non-negative, got {warmup!r}")

    for _ in range(warmup):
        workload()

    samples = []
    for _ in range(runs):
        start = time.perf_counter()
        workload()
        samples.append(time.perf_counter() - start)

    return median(samples)


# ━━━━━━  Notebook 1 of 5 · WATCH · Vector add in CuPy RawKernel (speaker demo)  ━━━━━━

# 01 · Vector add in CuPy RawKernel - _speaker demo_

**Goal:** see what a GPU kernel actually is - a string of CUDA C, JIT-compiled and launched from Python.

This is the only CuPy moment in the workshop. Everything after this is in Triton.

## The kernel - CUDA C as a Python string

Everything between the triple quotes is real CUDA C compiled at runtime. Things to notice:

- `__global__` marks a function callable from the host (Python) and run on the GPU.
- `blockIdx.x * blockDim.x + threadIdx.x` is the universal “where am I in the grid?” idiom.
- The `if (i < n)` guard is why kernels need explicit masking - we will see Triton's `mask=` equivalent next.

In [ ]:
kernel_src = r'''
extern "C" __global__
void vec_add(const float* a, const float* b, float* c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];
}
'''

vec_add = cp.RawKernel(kernel_src, 'vec_add')

## Inputs

In [ ]:
n = 1 << 20  # one million elements
a = cp.random.random(n, dtype=cp.float32)
b = cp.random.random(n, dtype=cp.float32)
c = cp.empty_like(a)
print(f'n = {n:,} elements')

## Launch

In [ ]:
threads_per_block = 256
blocks_per_grid = (n + threads_per_block - 1) // threads_per_block

vec_add((blocks_per_grid,), (threads_per_block,), (a, b, c, n))
cp.cuda.runtime.deviceSynchronize()
print(f'launched {blocks_per_grid:,} blocks x {threads_per_block} threads = {blocks_per_grid * threads_per_block:,} threads total')

## Correctness

In [ ]:
cp.testing.assert_allclose(c, a + b)
print('correctness: ok')

## Timing (a quick measurement)

In [ ]:
start = cp.cuda.Event(); end = cp.cuda.Event()
# warmup
for _ in range(3):
    vec_add((blocks_per_grid,), (threads_per_block,), (a, b, c, n))
cp.cuda.runtime.deviceSynchronize()

start.record()
for _ in range(100):
    vec_add((blocks_per_grid,), (threads_per_block,), (a, b, c, n))
end.record()
end.synchronize()
ms = cp.cuda.get_elapsed_time(start, end) / 100
bandwidth_gbs = (3 * a.nbytes) / (ms * 1e-3) / 1e9  # read a, read b, write c
print(f'avg kernel time: {ms:.3f} ms')
print(f'effective bandwidth: {bandwidth_gbs:.1f} GB/s')

## Teaching beats

1. The CUDA C string is the **only** place anything truly “GPU” exists. Everything else is launcher glue.
2. `blockIdx.x * blockDim.x + threadIdx.x` is the universal “where am I?” idiom - circle it twice.
3. The `if (i < n)` guard is why kernels need explicit masking. We will see Triton's `mask=` equivalent in the next notebook.
4. Compare `kernel((blocks,), (threads,), ...)` to the equivalent CPU loop - same algorithm, different launcher.

On to Triton.

# ━━━━━━  Notebook 2 of 5 · WRITE · Vector add in Triton (1 TODO)  ━━━━━━

# 02 · Vector add in Triton - _solution_

Reference solution for the one blank (the `mask`). Released after the workshop ends.

## Inputs and reference output

In [ ]:
n = 1 << 20
a = torch.rand(n, device='cuda', dtype=torch.float32)
b = torch.rand(n, device='cuda', dtype=torch.float32)
reference = a + b  # torch's answer; we will compare to it

## The kernel

One blank - the `mask`. Read the comment above it carefully; it tells you exactly what to write. The other lines (offsets, then load/add/store) are already filled in so you can see the pattern.

Helpful Triton API reminders:
- `tl.program_id(axis=0)` returns the integer id of this kernel instance along axis 0.
- `tl.arange(0, BLOCK_SIZE)` is a vector `[0, 1, 2, ..., BLOCK_SIZE-1]`.
- `tl.load(ptr + offsets, mask=mask)` loads a vector with masked-off elements set to 0 by default.
- `tl.store(ptr + offsets, value, mask=mask)` writes back the masked elements.

![diagram](https://raw.githubusercontent.com/abhiksark/pycon-italy-2026-workshop/main/notebooks/shared/diagrams/mask-load-1d.png)

![diagram](https://raw.githubusercontent.com/abhiksark/pycon-italy-2026-workshop/main/notebooks/shared/diagrams/tile-1d.png)

Mental picture: four programs partition the input. The last program's tail is hidden by the mask.

![diagram](https://raw.githubusercontent.com/abhiksark/pycon-italy-2026-workshop/main/notebooks/shared/diagrams/nb02-vector-add.png)

In [ ]:
@triton.jit
def vec_add_kernel(
    a_ptr, b_ptr, out_ptr,
    n,
    BLOCK_SIZE: tl.constexpr,
):
    pid = tl.program_id(axis=0)

    # per-program offsets (given)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)

    # (1/1) mask the tail block
    mask = offsets < n

    # load, add, store (given)
    a = tl.load(a_ptr + offsets, mask=mask)
    b = tl.load(b_ptr + offsets, mask=mask)
    tl.store(out_ptr + offsets, a + b, mask=mask)

## Launch

In [ ]:
out = torch.empty_like(a)
BLOCK_SIZE = 1024
grid = (triton.cdiv(n, BLOCK_SIZE),)
vec_add_kernel[grid](a, b, out, n, BLOCK_SIZE=BLOCK_SIZE)

## Correctness

In [ ]:
torch.testing.assert_close(out, reference)
print('correctness: ok')

## Quick timing vs `torch.add`

In [ ]:
import time

def time_op(fn, runs=50, warmup=5):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(runs):
        fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - start) / runs

def run_triton():
    vec_add_kernel[grid](a, b, out, n, BLOCK_SIZE=BLOCK_SIZE)

def run_torch():
    torch.add(a, b, out=out)

triton_ms = time_op(run_triton) * 1e3
torch_ms = time_op(run_torch) * 1e3
print(f'triton: {triton_ms:.3f} ms | torch: {torch_ms:.3f} ms | ratio: {triton_ms / torch_ms:.2f}x torch')

# --- effective bandwidth: read a, read b, write out ---
n_bytes = 3 * a.numel() * a.element_size()
gbs = n_bytes / (triton_ms * 1e-3) / 1e9
T4_PEAK_HBM_GBS = 320.0       # T4 datasheet
print(f'effective bandwidth: {gbs:.1f} GB/s ({gbs / T4_PEAK_HBM_GBS * 100:.0f}% of T4 peak HBM)')


![diagram](https://raw.githubusercontent.com/abhiksark/pycon-italy-2026-workshop/main/notebooks/shared/diagrams/compile-step-4.png)

## Under the hood: `@triton.jit` and the launcher

You just launched a kernel and got the right answer, but Python did not run the kernel - it **launched** it. Here is what `@triton.jit` plus `kernel[grid](...)` actually does:

1. `@triton.jit` captures the function's **source / AST** at import time. It does not execute it.
2. The first time you call `kernel[grid](a, b, out, n, BLOCK_SIZE=1024)`, Triton lowers it through:
   - **Triton-IR** - the tile-level IR, close to what you wrote.
   - **TritonGPU-IR** - adds GPU specifics: tensor layouts, register vs. shared memory.
   - **LLVM IR** - scalar.
   - **PTX** - NVIDIA's virtual ISA.
   - The CUDA driver JIT-compiles PTX to **SASS** (the real machine code for *your* GPU) and loads the resulting `cubin`.
3. The result is **cached** in `~/.triton/cache/` (override with `TRITON_CACHE_DIR`). The cache key includes every argument's dtype, every `tl.constexpr` value (e.g. `BLOCK_SIZE`), pointer alignment hints (multiples of 16 by default), and any integer args specialized as `equal_to_1`. Change any of those - fresh compile.
4. `kernel[grid](...)` is the **launcher**. It computes the cache key, fetches or compiles the artifact, packs the args into the calling convention, and calls `cuLaunchKernel` via the driver. It returns the `CompiledKernel` so you can inspect every IR stage.

Two side notes:

- **Autotune.** `@triton.autotune` compiles **one specialization per config** and benchmarks them on the first call to pick the fastest. Each config caches independently. You will see this in nb05.
- **Dumping IR for offline reading.** Run your script with `TRITON_KERNEL_DUMP=1 MLIR_ENABLE_DUMP=1 LLVM_IR_ENABLE_DUMP=1` and every stage hits the filesystem.

Let's actually look at what compiled, and prove the cache is real.

In [ ]:
# kernel[grid](...) returns the CompiledKernel for this launch.
# Re-launch (cache hit) to grab the handle without recompiling.
compiled = vec_add_kernel[grid](a, b, out, n, BLOCK_SIZE=BLOCK_SIZE)

print('IR stages held on this object:', list(compiled.asm.keys()))
print()
print('--- TTIR (Triton IR, tile-level - close to what you wrote) ---')
print(compiled.asm['ttir'])
print('--- PTX (first 30 lines - NVIDIA virtual ISA) ---')
print('\n'.join(compiled.asm['ptx'].splitlines()[:30]))

In [ ]:
# Triton caches one compiled artifact per (dtype, constexpr, alignment, ...) key.
# JITFunction.device_caches is keyed by device id; each value is a 5-tuple whose [0] holds the specialization dict.
def n_specializations():
    return sum(len(v[0]) for v in vec_add_kernel.device_caches.values())

before = n_specializations()

# New BLOCK_SIZE -> new specialization -> fresh compile.
out_alt = torch.empty_like(a)
BLOCK_ALT = 512
grid_alt = (triton.cdiv(n, BLOCK_ALT),)
vec_add_kernel[grid_alt](a, b, out_alt, n, BLOCK_SIZE=BLOCK_ALT)
mid = n_specializations()

# Same key again -> cache hit, no growth.
vec_add_kernel[grid_alt](a, b, out_alt, n, BLOCK_SIZE=BLOCK_ALT)
after = n_specializations()

print(f'specializations cached: start={before}  after_new_BLOCK_SIZE={mid}  after_repeat={after}')
print('correctness with BLOCK_SIZE=512:', torch.allclose(out_alt, a + b))

## What just happened

You wrote a GPU kernel without writing CUDA C. The grid is one-dimensional. Each program handles one BLOCK_SIZE-long chunk. The mask handles the tail when `n` is not a multiple of `BLOCK_SIZE`.

Next up: reductions - the canonical “sum a big tensor” kernel that introduces shared-memory thinking.

In [ ]:
# Bonus — when the roofline lies.
#
# The roofline says a streaming kernel like ours should sit on HBM peak.
# That's true at the limit. But at small N the kernel finishes faster than
# HBM has time to deliver peak — the bottleneck is launch overhead, not
# bandwidth. Watch where the kernel crosses over.

sizes = [(1 << e) for e in (10, 14, 18, 20, 22, 24)]
print(f'{"N":>12}  {"buf":>7}  {"time":>10}  {"GB/s":>8}  {"%T4 peak":>9}')
print('-' * 56)
for n_b in sizes:
    a_b   = torch.randn(n_b, device='cuda')
    b_b   = torch.randn(n_b, device='cuda')
    out_b = torch.empty_like(a_b)
    grid_b = (triton.cdiv(n_b, 1024),)

    def run():
        vec_add_kernel[grid_b](a_b, b_b, out_b, n_b, BLOCK_SIZE=1024)
        torch.cuda.synchronize()

    runs = 500 if n_b < (1 << 16) else 100 if n_b < (1 << 22) else 30
    secs = median_seconds(run, runs=runs, warmup=20)
    bytes_moved = 3 * n_b * a_b.element_size()        # read a, read b, write out
    gbs = bytes_moved / secs / 1e9
    pct = gbs / T4_PEAK_HBM_GBS * 100
    unit, val = ('us', secs * 1e6) if secs < 1e-3 else ('ms', secs * 1e3)
    print(f'{n_b:>12,}  {n_b*4/1e6:>5.2f}MB  {val:>7.1f} {unit}  {gbs:>7.1f}  {pct:>7.0f}%')

print(
    "\nReading the table:"
    "\n  Tiny N — bandwidth looks ~0%. The kernel finishes before HBM has time"
    "\n    to deliver peak; the bottleneck is launch overhead, not bandwidth."
    "\n    The roofline has no axis for launch latency, so in this regime it"
    "\n    lies — it reports 'memory-bound at a low ceiling' instead of"
    "\n    'parallelism-bound, model does not apply.'"
    "\n  Large N — bandwidth converges to a real, measurable HBM ceiling."
    "\n    THIS is the regime where the roofline tells the truth."
    "\n  '% of T4 peak >100%' just means this card's HBM is faster than a T4's."
)


# ━━━━━━  Notebook 3 of 5 · WRITE · Reduction in Triton (2 TODOs)  ━━━━━━

# 03 · Reduction in Triton - _solution_

Reference solutions for the core (single-block sum) and bonus (multi-block sum) TODOs.

# Part 1 · Core: single-block sum

We start with a tensor short enough to fit in one block (BLOCK_SIZE = 1024). One program instance sums everything. `tl.sum(x, axis=0)` does the tree reduction across the block.

Two TODOs. Read the comments.

Mental picture: `tl.sum` collapses the block via a hidden binary tree.

![diagram](https://raw.githubusercontent.com/abhiksark/pycon-italy-2026-workshop/main/notebooks/shared/diagrams/nb03-reduction-tree.png)

![diagram](https://raw.githubusercontent.com/abhiksark/pycon-italy-2026-workshop/main/notebooks/shared/diagrams/mask-load-1d.png)

![diagram](https://raw.githubusercontent.com/abhiksark/pycon-italy-2026-workshop/main/notebooks/shared/diagrams/tile-1d.png)

In [ ]:
@triton.jit
def single_block_sum(x_ptr, out_ptr, n, BLOCK_SIZE: tl.constexpr):
    offsets = tl.arange(0, BLOCK_SIZE)
    mask = offsets < n
    x = tl.load(x_ptr + offsets, mask=mask, other=0.0)
    total = tl.sum(x, axis=0)
    tl.store(out_ptr, total)


In [ ]:
n = 1000
x = torch.rand(n, device='cuda', dtype=torch.float32)
out = torch.empty(1, device='cuda', dtype=torch.float32)
single_block_sum[(1,)](x, out, n, BLOCK_SIZE=1024)
torch.testing.assert_close(out[0], x.sum(), rtol=1e-3, atol=1e-3)
print(f'core ok | sum = {out.item():.4f}')

# Part 2 · Challenge: multi-block sum

Tensors bigger than BLOCK_SIZE need more than one program. Each program sums its own block into a per-block partial. Then we sum the partials on the host (a tiny CPU reduction) - or with a second kernel call (left as homework).

Two TODOs.

In [ ]:
@triton.jit
def multi_block_sum(x_ptr, partials_ptr, n, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(axis=0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n
    x = tl.load(x_ptr + offsets, mask=mask, other=0.0)
    block_sum = tl.sum(x, axis=0)
    tl.store(partials_ptr + pid, block_sum)


In [ ]:
n_big = 1 << 20
x_big = torch.rand(n_big, device='cuda', dtype=torch.float32)
BLOCK = 1024
grid = (triton.cdiv(n_big, BLOCK),)
partials = torch.empty(grid[0], device='cuda', dtype=torch.float32)
multi_block_sum[grid](x_big, partials, n_big, BLOCK_SIZE=BLOCK)
result = partials.sum()  # host-side final reduction (one transfer, one sum)
torch.testing.assert_close(result, x_big.sum(), rtol=1e-2, atol=1e-2)
print(f'bonus ok | sum = {result.item():.2f}')

In [ ]:
# --- benchmark: how close to peak HBM are we? ---

def run_reduction():
    multi_block_sum[grid](x_big, partials, n_big, BLOCK_SIZE=BLOCK)
    partials.sum()
    torch.cuda.synchronize()

ms = median_seconds(run_reduction, runs=50, warmup=5) * 1e3
n_bytes = x_big.numel() * x_big.element_size()  # one streaming read of the input dominates
gbs = n_bytes / (ms * 1e-3) / 1e9
T4_PEAK_HBM_GBS = 320.0       # T4 datasheet
print(f'reduction: {ms:.3f} ms | {gbs:.1f} GB/s ({gbs / T4_PEAK_HBM_GBS * 100:.0f}% of T4 peak HBM)')


## Teaching beat

Reductions on a GPU are tree-shaped, not linear. `tl.sum` hides the tree from you, but it is there - every warp does a local reduction in registers, then warps cooperate via shared memory.

Next: 2D, with halos.

# ━━━━━━  Notebook 4 of 5 · MEASURE · Fused softmax + image blur (5 TODOs)  ━━━━━━

# 04 · Softmax & blur in Triton - _solution_

Reference solutions for Part 1 (fused softmax) and the Part 3 bonus (2D blur with halos). Part 2 (1D blur) is a worked example in the attendee notebook too.

# Part 1 · Fused softmax: collapse the memory wall

Part 3 set up the memory wall. **Naive softmax** in three passes:

```python
m = x.max(dim=1, keepdim=True).values
e = (x - m).exp()
out = e / e.sum(dim=1, keepdim=True)
```

Three kernel launches. ~6N bytes of HBM traffic for ~3N FLOPs → **AI ≈ ½**. Sits well below the ridge.

**Fused softmax** does the whole thing in one kernel - one read of `x`, one write of `out`, ~2N bytes. One program per row; the whole row lives in registers while we do max → `exp(x − m)` → sum → divide.

In [ ]:
n_rows, n_cols = 1823, 1200
x = torch.randn(n_rows, n_cols, device='cuda', dtype=torch.float32)
y = torch.empty_like(x)
reference = torch.softmax(x, dim=1)
print(f'input: ({n_rows}, {n_cols})')

In [ ]:
@triton.jit
def fused_softmax_kernel(
    x_ptr, y_ptr,
    n_cols,
    BLOCK_SIZE: tl.constexpr,
):
    row = tl.program_id(axis=0)
    cols = tl.arange(0, BLOCK_SIZE)
    mask = cols < n_cols
    row_offset = row * n_cols

    x = tl.load(x_ptr + row_offset + cols, mask=mask, other=-float('inf'))
    m = tl.max(x, axis=0)
    e = tl.exp(x - m)
    s = tl.sum(e, axis=0)
    out = e / s
    tl.store(y_ptr + row_offset + cols, out, mask=mask)

In [ ]:
BLOCK_SIZE = triton.next_power_of_2(n_cols)  # 2048 for n_cols=1200
grid = (n_rows,)
fused_softmax_kernel[grid](x, y, n_cols, BLOCK_SIZE=BLOCK_SIZE)
torch.testing.assert_close(y, reference, rtol=1e-5, atol=1e-5)
print('correctness: ok')

## Naive vs fused - the payoff Part 3 promised

In [ ]:

def naive_softmax(x):
    m = x.max(dim=1, keepdim=True).values
    e = (x - m).exp()
    return e / e.sum(dim=1, keepdim=True)

def run_triton():
    fused_softmax_kernel[grid](x, y, n_cols, BLOCK_SIZE=BLOCK_SIZE)
    torch.cuda.synchronize()

def run_naive():
    naive_softmax(x)
    torch.cuda.synchronize()

t_triton = median_seconds(run_triton, runs=50, warmup=5) * 1e3
t_naive  = median_seconds(run_naive,  runs=50, warmup=5) * 1e3

n_bytes_fused = 2 * x.numel() * x.element_size()
n_bytes_naive = 6 * x.numel() * x.element_size()
gbs_triton = n_bytes_fused / (t_triton * 1e-3) / 1e9
gbs_naive  = n_bytes_naive / (t_naive  * 1e-3) / 1e9

T4_PEAK_HBM_GBS = 320.0
print(f'naive PyTorch (3 passes): {t_naive:7.3f} ms | {gbs_naive:6.1f} GB/s effective ({gbs_naive/T4_PEAK_HBM_GBS*100:.0f}% T4 peak)')
print(f'fused Triton  (1 pass) :  {t_triton:7.3f} ms | {gbs_triton:6.1f} GB/s effective ({gbs_triton/T4_PEAK_HBM_GBS*100:.0f}% T4 peak)')
print(f'speedup: {t_naive / t_triton:.2f}x by collapsing three passes into one kernel.')

# Part 2 · Image blur (worked example)

*Bologna's Two Towers (Asinelli + Garisenda) - photo by Volodymyr Vlasenko, [CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/), [via Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Asinelli_Tower_and_Garisenda_Tower_Bologna_Italy.jpg). 2944x4444 ≈ 13 MP.*

In [ ]:
src = Image.open('notebooks/shared/assets/blur-source-bologna.jpg').convert('RGB')
src_np = np.asarray(src, dtype=np.float32) / 255.0  # (H, W, 3); H=4444, W=2944
H, W, _ = src_np.shape
# Operate on the luminance channel for simplicity. We will blur a single channel.
lum = src_np.mean(axis=2)  # (H, W) float32 in [0, 1]
lum_t = torch.from_numpy(lum).cuda().contiguous()
out_t = torch.empty_like(lum_t)
print(f'image {H}x{W}')

## 1D horizontal blur (one program per row)

Each program handles one row. For every pixel `(row, col)` in the row, we compute the average of the three pixels at columns `col-1`, `col`, `col+1`. The lookups at `col-1` and `col+1` need masked **halos** when they step off the row.

In [ ]:
@triton.jit
def blur_1d(in_ptr, out_ptr, H, W, BLOCK_W: tl.constexpr):
    row = tl.program_id(axis=0)
    cols = tl.arange(0, BLOCK_W)
    mask = cols < W
    base = row * W
    left = tl.load(in_ptr + base + (cols - 1), mask=(cols - 1 >= 0) & mask, other=0.0)
    centre = tl.load(in_ptr + base + cols, mask=mask, other=0.0)
    right = tl.load(in_ptr + base + (cols + 1), mask=(cols + 1 < W) & mask, other=0.0)
    tl.store(out_ptr + base + cols, (left + centre + right) / 3.0, mask=mask)


In [ ]:
BLOCK_W = triton.next_power_of_2(W)
grid = (H,)
blur_1d[grid](lum_t, out_t, H, W, BLOCK_W=BLOCK_W)

# Compare with a numpy reference
import numpy as np
ref = np.zeros_like(lum)
ref[:, 1:-1] = (lum[:, :-2] + lum[:, 1:-1] + lum[:, 2:]) / 3.0
ref[:, 0] = (lum[:, 0] + lum[:, 1]) / 3.0
ref[:, -1] = (lum[:, -2] + lum[:, -1]) / 3.0
torch.testing.assert_close(out_t, torch.from_numpy(ref).cuda(), rtol=1e-3, atol=1e-3)
print('core ok')

## Look at the result

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(lum, cmap='gray'); axes[0].set_title('original')
axes[1].imshow(out_t.cpu().numpy(), cmap='gray'); axes[1].set_title('1D-blurred')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

# Part 3 · Bonus challenge: 2D blur with halos

Same idea but in 2D: every pixel averages its 3x3 neighbourhood. We tile the output into `(BLOCK_M, BLOCK_N)` blocks; each program handles one tile and loads a tile-plus-halo from the input.

Mental picture: each output tile reads a 1-pixel halo. Halo cells outside the image are masked to 0.

![diagram](https://raw.githubusercontent.com/abhiksark/pycon-italy-2026-workshop/main/notebooks/shared/diagrams/nb04-blur-halo.png)

![diagram](https://raw.githubusercontent.com/abhiksark/pycon-italy-2026-workshop/main/notebooks/shared/diagrams/pointer-grid-2d.png)

In [ ]:
@triton.jit
def blur_2d(in_ptr, out_ptr, H, W,
            BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    pid_m = tl.program_id(axis=0)
    pid_n = tl.program_id(axis=1)
    row_offsets = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    col_offsets = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    row_mask = row_offsets < H
    col_mask = col_offsets < W

    total = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for dy in tl.static_range(-1, 2):
        for dx in tl.static_range(-1, 2):
            r = row_offsets[:, None] + dy
            c = col_offsets[None, :] + dx
            m = (r >= 0) & (r < H) & (c >= 0) & (c < W)
            v = tl.load(in_ptr + r * W + c, mask=m, other=0.0)
            total += v

    out_ptrs = out_ptr + row_offsets[:, None] * W + col_offsets[None, :]
    tl.store(out_ptrs, total / 9.0, mask=row_mask[:, None] & col_mask[None, :])


In [ ]:
out_t2 = torch.empty_like(lum_t)
BLOCK_M, BLOCK_N = 32, 32
assert lum_t.is_contiguous(), 'blur_2d requires a contiguous tensor'
grid = (triton.cdiv(H, BLOCK_M), triton.cdiv(W, BLOCK_N))
blur_2d[grid](lum_t, out_t2, H, W, BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N)

# Reference via padding + slicing
pad = np.pad(lum, 1, mode='constant')
ref2 = sum(
    pad[1+dy:1+dy+H, 1+dx:1+dx+W] for dy in (-1, 0, 1) for dx in (-1, 0, 1)
) / 9.0
torch.testing.assert_close(out_t2, torch.from_numpy(ref2.astype(np.float32)).cuda(), rtol=1e-3, atol=1e-3)
print('bonus ok')

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(lum, cmap='gray'); axes[0].set_title('original')
axes[1].imshow(out_t2.cpu().numpy(), cmap='gray'); axes[1].set_title('2D blurred')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

## How fast is this? Triton vs. OpenCV

OpenCV's `cv2.blur` is the right baseline — a heavily SIMD-optimised, multi-threaded 3x3 box filter is what a CPU does at production speed. We time both with their natural tools (CUDA events for GPU, `time.perf_counter` for CPU), then print the speedup.

In [ ]:
# How fast is this? Triton vs. OpenCV
import statistics, time
import cv2

def run_triton_blur():
    blur_2d[grid](lum_t, out_t2, H, W, BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N)
    torch.cuda.synchronize()

triton_ms = median_seconds(run_triton_blur, runs=20, warmup=5) * 1000

# Effective HBM traffic: read the image once, write it once.
n_bytes = lum.nbytes * 2
gbs = n_bytes / (triton_ms / 1000) / 1e9

def run_cv2_blur():
    cv2.blur(lum, (3, 3))

for _ in range(3): run_cv2_blur()                          # warm up the CPU caches
cv2_runs = []
for _ in range(20):
    t0 = time.perf_counter()
    run_cv2_blur()
    cv2_runs.append(time.perf_counter() - t0)
cv2_ms = statistics.median(cv2_runs) * 1000

print(f'Triton blur_2d (T4):    {triton_ms:6.2f} ms   |   {gbs:5.0f} GB/s')
print(f'OpenCV cv2.blur (CPU):  {cv2_ms:6.2f} ms')
print(f'Speedup:                {cv2_ms / triton_ms:5.1f}x')

## Profile with Proton

Same kernel, this time through `triton.profiler` — emits a `.hatchet` trace you can open with `proton-viewer` for per-kernel attribution. The slide we just showed, in action.

In [ ]:
# Same kernel, profiled with triton.profiler. Writes blur.hatchet.
try:
    import triton.profiler as proton
    proton.start('blur')
    blur_2d[grid](lum_t, out_t2, H, W, BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N)
    torch.cuda.synchronize()
    proton.finalize()
    print('Wrote blur.hatchet')
    print('Inspect:   proton-viewer -m time/ms,gbyte/s blur.hatchet')
except (ImportError, AttributeError) as exc:
    print(f'Proton not available in this Triton build: {exc}')

## Teaching beat

Every load that goes outside the image boundary must be masked. This is the same pattern as the vector-add tail, generalised to 2D - just with **two** boundary checks instead of one.

# ━━━━━━  Notebook 5 of 5 · CLIMB · Tiled matmul in Triton (6 TODOs)  ━━━━━━

# 05 · Tiled matmul in Triton - _solution_

Reference solution for all six TODOs.

## Inputs and reference output

In [ ]:
M, N, K = 512, 512, 512  # small enough to iterate fast on a T4
A = torch.randn((M, K), device='cuda', dtype=torch.float32)
B = torch.randn((K, N), device='cuda', dtype=torch.float32)
reference = A @ B
print(f'A: {tuple(A.shape)} | B: {tuple(B.shape)} | C: ({M}, {N})')

## The kernel

Read the comments above every TODO. The structure of the kernel is given; you fill in the meaningful lines.

nb04's blur assumed a contiguous tensor and indexed with a bare `* W`. Real tensors carry **strides** - the element gap between consecutive rows and columns - so this kernel takes them explicitly (`stride_am`, `stride_ak`, ...) and never assumes a memory layout.

**Mental model:** program `(pid_m, pid_n)` is responsible for the `(BLOCK_M, BLOCK_N)` output tile at row-block `pid_m`, col-block `pid_n`. To compute that tile we accumulate `K // BLOCK_K` partial products of `(BLOCK_M, BLOCK_K) @ (BLOCK_K, BLOCK_N)` tiles.

![diagram](https://raw.githubusercontent.com/abhiksark/pycon-italy-2026-workshop/main/notebooks/shared/diagrams/nb05-kloop.png)

Mental picture: the K-loop sweeps a row of A and a column of B, accumulating their products into one tile of C.

![diagram](https://raw.githubusercontent.com/abhiksark/pycon-italy-2026-workshop/main/notebooks/shared/diagrams/nb05-matmul-tiling.png)

In [ ]:
# Canonical tiled matmul kernel — inlined from solutions/_shared.py so this combined notebook
# is self-contained (the public Colab clone has no solutions/ directory).
@triton.jit
def matmul_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
):
    """Computes C = A @ B as a tiled matmul. See nb05 for the pedagogical walk-through."""
    pid_m = tl.program_id(axis=0)
    pid_n = tl.program_id(axis=1)
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k_start in range(0, K, BLOCK_K):
        k_offs = k_start + offs_k
        a_ptrs = a_ptr + offs_m[:, None] * stride_am + k_offs[None, :] * stride_ak
        b_ptrs = b_ptr + k_offs[:, None] * stride_bk + offs_n[None, :] * stride_bn
        a_mask = (offs_m[:, None] < M) & (k_offs[None, :] < K)
        b_mask = (k_offs[:, None] < K) & (offs_n[None, :] < N)
        a = tl.load(a_ptrs, mask=a_mask, other=0.0)
        b = tl.load(b_ptrs, mask=b_mask, other=0.0)
        # input_precision='ieee' forces true fp32 even on Ampere+ where tl.dot
        # would otherwise use TF32 tensor cores. Costs nothing on T4 (no TF32);
        # keeps the kernel bit-portable so the 1e-3 tolerance holds anywhere.
        acc += tl.dot(a, b, input_precision='ieee')
    c_ptrs = c_ptr + offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn
    c_mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)
    tl.store(c_ptrs, acc, mask=c_mask)


![diagram](https://raw.githubusercontent.com/abhiksark/pycon-italy-2026-workshop/main/notebooks/shared/diagrams/nb05-store.png)

## Launch and correctness

In [ ]:
BLOCK_M = BLOCK_N = BLOCK_K = 64
C = torch.empty((M, N), device='cuda', dtype=torch.float32)

grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
matmul_kernel[grid](
    A, B, C,
    M, N, K,
    A.stride(0), A.stride(1),
    B.stride(0), B.stride(1),
    C.stride(0), C.stride(1),
    BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K,
)
torch.testing.assert_close(C, reference, rtol=1e-3, atol=1e-3)
print('correctness: ok')

## Benchmark

In [ ]:

def run_triton():
    matmul_kernel[grid](
        A, B, C, M, N, K,
        A.stride(0), A.stride(1),
        B.stride(0), B.stride(1),
        C.stride(0), C.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K,
    )
    torch.cuda.synchronize()

def run_torch():
    torch.matmul(A, B, out=C)
    torch.cuda.synchronize()

t_triton = median_seconds(run_triton, runs=20, warmup=5)
t_torch = median_seconds(run_torch, runs=20, warmup=5)
g_triton = gflops_for_matmul(M, N, K, t_triton)
g_torch = gflops_for_matmul(M, N, K, t_torch)
print(f'triton: {t_triton*1e3:7.3f} ms | {g_triton:8.1f} GFLOPs')
print(f'torch:  {t_torch*1e3:7.3f} ms | {g_torch:8.1f} GFLOPs')
print(f'ratio:  {g_triton / g_torch * 100:5.1f}% of cuBLAS')

# --- where on the roofline are we? ---
T4_PEAK_TFLOPS_FP32 = 8.1     # T4 datasheet, fp32, no tensor cores
T4_PEAK_HBM_GBS = 320.0       # T4 datasheet
machine_balance = T4_PEAK_TFLOPS_FP32 * 1e3 / T4_PEAK_HBM_GBS  # FLOPs/byte at the knee
ai = (2 * M * N * K) / (4 * (M*K + K*N + M*N))  # ideal AI: each matrix loaded once
peak_pct = (g_triton / 1e3) / T4_PEAK_TFLOPS_FP32 * 100
print(f'arithmetic intensity: {ai:.0f} FLOPs/byte (compute-bound > {machine_balance:.0f})')
print(f'achieved:             {g_triton/1e3:.2f} TFLOPS ({peak_pct:.0f}% of T4 peak fp32)')


## You did it

If `correctness: ok` printed and the ratio is somewhere in the 30–60% range on a T4, you have a working tiled matmul. The remaining 40–70% gap to `torch.matmul` is what autotuning, Tensor Cores, and `tl.dot` precision modes buy you - each one is a workshop of its own.

Run `bench/compare_vs_torch.py` to see the same kernel against a naive Python triple-loop and `numpy.matmul`. The naive-Python baseline is _five orders of magnitude_ slower than what you just wrote.

## Going further

This kernel is deliberately the *simple* tiled matmul. Four best-practice levers carry it toward `torch.matmul` - each is one line of intent and a chapter of depth:

| Lever | What it does | Typical win |
|-------|--------------|-------------|
| `@triton.autotune` | Compiles several `BLOCK_*` / `num_warps` / `num_stages` configs, benchmarks them on the first call, caches the winner per shape | the best tile for *your* GPU |
| `num_stages` pipelining | Overlaps the next K-tile's HBM load with the current tile's `tl.dot` - software pipelining | hides memory latency |
| `GROUP_SIZE_M` swizzling | Re-orders program IDs so neighbouring output tiles reuse the A/B rows already sitting in L2 | ~1.3x, +60% L2 hit rate |
| Tensor Cores (TF32) | Drop `input_precision='ieee'` and Ampere+ runs `tl.dot` on TF32 tensor cores | several x - at reduced precision |

The official [Triton matmul tutorial](https://triton-lang.org/main/getting-started/tutorials/03-matrix-multiplication.html) layers all four onto exactly this kernel. Start with `@triton.autotune` - highest reward for the least code.